# HC-DPO: analysis

Reads `metrics.json` and `train_log.jsonl` from every run folder under `OUTPUT_DIR`. It does not need W&B.

The claim under test is control: HC-DPO should reach a chosen KL in one run. So the calibration plot is the main result, and reward vs KL is the check that this control costs no reward.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# On Colab, mount Drive first and point this at the project's outputs folder.
# from google.colab import drive; drive.mount('/content/drive')
OUTPUT_DIR = Path('/content/drive/MyDrive/Projects/hc-dpo/outputs')

In [ ]:
def load_metrics(output_dir: Path) -> pd.DataFrame:
    """One row per evaluated run."""
    rows = [json.loads(path.read_text()) for path in sorted(output_dir.glob('*/metrics.json'))]
    return pd.DataFrame(rows)


def load_train_log(run_dir: Path) -> pd.DataFrame:
    """All logged training metrics of one run."""
    lines = (run_dir / 'train_log.jsonl').read_text().splitlines()
    return pd.DataFrame([json.loads(line) for line in lines if line.strip()])


metrics = load_metrics(OUTPUT_DIR)
hcdpo_runs = metrics[metrics['method'] == 'hcdpo']
columns = ['run_name', 'method', 'beta', 'epsilon', 'seed', 'kl_mean', 'kl_stderr', 'reward_mean', 'reward_stderr',
           'reward_diff_vs_base_mean', 'reward_diff_vs_base_stderr', 'preference_accuracy', 'kl_truncated_fraction']
metrics[[c for c in columns if c in metrics.columns]]

## 1. Calibration: final KL vs target epsilon (main result)

Each point is one HC-DPO run. Points on the dashed line hit their KL budget exactly.

In [ ]:
if len(hcdpo_runs) == 0:
    print('No HC-DPO runs yet.')
else:
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.errorbar(hcdpo_runs['epsilon'].astype(float), hcdpo_runs['kl_mean'], yerr=hcdpo_runs['kl_stderr'],
                fmt='o', capsize=3, label='HC-DPO runs')
    limit = 1.1 * max(hcdpo_runs['epsilon'].astype(float).max(), hcdpo_runs['kl_mean'].max())
    ax.plot([0, limit], [0, limit], linestyle='--', color='gray', label='final KL = epsilon')
    ax.set_xlabel('Target epsilon (nats per response)')
    ax.set_ylabel('Final KL(policy || reference)')
    ax.set_title('Calibration')
    ax.legend()
    plt.show()

## 2. Reward vs KL

Each point is one run: final evaluation KL (300 test prompts) against mean reward-model score. Error bars are one standard error. If HC-DPO's control costs nothing, its points lie on the same curve as DPO's.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for method, group in metrics.groupby('method'):
    ax.errorbar(group['kl_mean'], group['reward_mean'], xerr=group['kl_stderr'], yerr=group['reward_stderr'],
                fmt='o', capsize=3, label=method)
    for _, row in group.iterrows():
        if method == 'dpo':
            text = f"beta={row['beta']}"
        elif method == 'hcdpo':
            text = f"eps={row['epsilon']}"
        else:
            continue
        ax.annotate(text, (row['kl_mean'], row['reward_mean']), textcoords='offset points', xytext=(6, 6), fontsize=8)
ax.set_xlabel('KL(policy || reference), nats per response')
ax.set_ylabel('Mean reward-model score')
ax.set_title('Reward vs KL')
ax.legend()
plt.show()

## 3. KL over training

KL estimated on the 32 fixed monitoring prompts during training; the band is one standard error. Dotted lines mark each HC-DPO run's epsilon. HC-DPO runs should level off near their epsilon; DPO runs end wherever their beta puts them.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for _, run in metrics[~metrics['is_base']].iterrows():
    log = load_train_log(OUTPUT_DIR / run['run_name'])
    kl = log.dropna(subset=['kl/mean'])
    line, = ax.plot(kl['step'], kl['kl/mean'], label=run['run_name'])
    ax.fill_between(kl['step'], kl['kl/mean'] - kl['kl/stderr'], kl['kl/mean'] + kl['kl/stderr'], alpha=0.2)
    if run['method'] == 'hcdpo':
        ax.axhline(float(run['epsilon']), linestyle=':', color=line.get_color())
ax.set_xlabel('Optimizer step')
ax.set_ylabel('KL(policy || reference), nats per response')
ax.set_title('KL over training')
ax.legend(fontsize=7)
plt.show()

## 4. Lambda over training (HC-DPO)

The multiplier that takes the place of beta. It should settle once the KL sits at epsilon; a line pinned at the top or bottom of its range means the constraint could not be met.

In [ ]:
if len(hcdpo_runs) == 0:
    print('No HC-DPO runs yet.')
else:
    fig, ax = plt.subplots(figsize=(7, 4))
    overshoot = []
    for _, run in hcdpo_runs.iterrows():
        log = load_train_log(OUTPUT_DIR / run['run_name']).dropna(subset=['hcdpo/lambda'])
        ax.plot(log['step'], log['hcdpo/lambda'], label=run['run_name'])
        last = log.iloc[-1]
        overshoot.append({'run_name': run['run_name'], 'epsilon': run['epsilon'], 'final_lambda': last['hcdpo/lambda'],
                          'max_overshoot': last['hcdpo/max_overshoot'], 'infeasible_fraction': last['hcdpo/infeasible_fraction']})
    ax.set_yscale('log')
    ax.set_xlabel('Optimizer step')
    ax.set_ylabel('lambda')
    ax.set_title('Lambda over training')
    ax.legend(fontsize=7)
    plt.show()
    display(pd.DataFrame(overshoot))

## 5. Matched-KL comparison across seeds

Mean and standard deviation over seeds for every setting that has at least two seeds (the best DPO beta, and HC-DPO with epsilon set to that run's final KL).

In [ ]:
runs = metrics[~metrics['is_base']].copy()
runs['setting'] = [f"beta={r['beta']}" if r['method'] == 'dpo' else f"epsilon={r['epsilon']}" for _, r in runs.iterrows()]
table = runs.groupby(['method', 'setting'])[['kl_mean', 'reward_mean', 'preference_accuracy']].agg(['mean', 'std', 'count'])
table[table[('kl_mean', 'count')] >= 2]